## 1. NIFTY 50 Constituent Universe

Source: NSE India
Snapshot date: 2026-10-28

The constituent list is frozen for this project.
The use of a current constituent list for historical
analysis introduces survivorship bias.

In [2]:
from pathlib import Path
from io import BytesIO
import json
import requests
import pandas as pd
from datetime import datetime
import yfinance as yf

In [ ]:
# Project paths
PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "universe"
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Official NSE constituent CSV
URL = "https://nsearchives.nseindia.com/content/indices/ind_nifty50list.csv"

response = requests.get(
    URL,
    headers={
        "User-Agent": "Mozilla/5.0",
        "Accept": "text/csv,*/*",
    },
    timeout=30,
)

response.raise_for_status()

# Preserve the original downloaded file unchanged
raw_path = RAW_DIR / "nifty50_constituents_2026-10-28.csv"
raw_path.write_bytes(response.content)

# Read a separate in-memory copy for validation
constituents = pd.read_csv(BytesIO(response.content))

# Basic validation
assert "Symbol" in constituents.columns
assert len(constituents) == 50
assert constituents["Symbol"].notna().all()
assert constituents["Symbol"].is_unique

# Create Yahoo Finance tickers separately
constituents["Yahoo_Ticker"] = (
    constituents["Symbol"].astype(str) + ".NS"
)

# Record source metadata
metadata = {
    "source": "NSE India",
    "source_url": URL,
    "snapshot_date": "2026-10-28",
    "retrieved_at": datetime.now().astimezone().isoformat(),
    "constituent_count": len(constituents),
    "raw_file": raw_path.name,
    "survivorship_bias": True,
}

metadata_path = RAW_DIR / "nifty50_universe_metadata.json"

with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=4)

print("Constituent count:", len(constituents))
print("Unique symbols:", constituents["Symbol"].nunique())
print("Raw file:", raw_path)
print("Metadata:", metadata_path)

display(
    constituents[
        ["Company Name", "Symbol", "Yahoo_Ticker"]
    ].head(10)
)

Constituent count: 50
Unique symbols: 50
Raw file: /Users/srisuphachawla/Documents/DataScience_Projects/Indian_Equity_Market_Risk_Forecaster/data/raw/universe/nifty50_constituents_2026-10-28.csv
Metadata: /Users/srisuphachawla/Documents/DataScience_Projects/Indian_Equity_Market_Risk_Forecaster/data/raw/universe/nifty50_universe_metadata.json


,Company Name,Symbol,Yahoo_Ticker
0,Adani Enterprises Ltd.,ADANIENT,ADANIENT.NS
1,Adani Ports and Special Economic Zone Ltd.,ADANIPORTS,ADANIPORTS.NS
2,Apollo Hospitals Enterprise Ltd.,APOLLOHOSP,APOLLOHOSP.NS
3,Asian Paints Ltd.,ASIANPAINT,ASIANPAINT.NS
4,Axis Bank Ltd.,AXISBANK,AXISBANK.NS
5,BSE Ltd.,BSE,BSE.NS
6,Bajaj Auto Ltd.,BAJAJ-AUTO,BAJAJ-AUTO.NS
7,Bajaj Finance Ltd.,BAJFINANCE,BAJFINANCE.NS
8,Bajaj Finserv Ltd.,BAJAJFINSV,BAJAJFINSV.NS
9,Bharat Electronics Ltd.,BEL,BEL.NS


In [ ]:
current_dir = Path.cwd()
PROJECT_ROOT = next(
    (
        path for path in [current_dir, *current_dir.parents]
        if (path / "PROTOCOL.md").exists()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate project root.")

UNIVERSE_DIR = PROJECT_ROOT / "data" / "raw" / "universe"

STOCK_RAW_DIR = PROJECT_ROOT / "data" / "raw" / "stocks"
MACRO_RAW_DIR = PROJECT_ROOT / "data" / "raw" / "market_macro"

STOCK_RAW_DIR.mkdir(parents=True, exist_ok=True)
MACRO_RAW_DIR.mkdir(parents=True, exist_ok=True)


# 2. LOAD FROZEN STOCK UNIVERSE
universe_file = (
    UNIVERSE_DIR / "nifty50_constituents_2026-10-28.csv"
)

universe = pd.read_csv(universe_file)

stock_tickers = (
    universe["Symbol"].astype(str) + ".NS"
).tolist()

macro_tickers = {
    "^NSEI": "NIFTY_50",
    "^INDIAVIX": "INDIA_VIX",
    "INR=X": "USD_INR",
    "BZ=F": "BRENT_CRUDE",
    "^TNX": "US_10Y_YIELD"
}

print("Stocks:", len(stock_tickers))
print("Additional market series:", len(macro_tickers))


# 3. DOWNLOAD FUNCTION
START_DATE = "2010-01-01"

download_log = []
def download_batch(tickers, output_dir, asset_type):

    data = yf.download(
        tickers=tickers,
        start=START_DATE,
        interval="1d",
        auto_adjust=False,
        actions=True,
        group_by="ticker",
        threads=True,
        progress=False,
        multi_level_index=True
    )

    for ticker in tickers:

        try:
            ticker_data = data[ticker].copy()

            if ticker_data.empty or ticker_data.isna().all().all():
                raise ValueError("No usable data returned.")

            # Preserve the downloaded observations.
            safe_name = (
                ticker.replace("^", "")
                .replace("=", "_")
                .replace(".", "_")
            )

            output_path = output_dir / f"{safe_name}.parquet"

            ticker_data.to_parquet(output_path)

            download_log.append({
                "ticker": ticker,
                "asset_type": asset_type,
                "status": "success",
                "rows": len(ticker_data),
                "first_date": ticker_data.index.min(),
                "last_date": ticker_data.index.max(),
                "file": str(output_path)
            })

        except Exception as error:

            download_log.append({
                "ticker": ticker,
                "asset_type": asset_type,
                "status": "failed",
                "rows": 0,
                "first_date": None,
                "last_date": None,
                "file": None,
                "error": str(error)
            })

# 4. DOWNLOAD STOCK DATA IN BATCHES
BATCH_SIZE = 10

for start in range(0, len(stock_tickers), BATCH_SIZE):
    batch = stock_tickers[start:start + BATCH_SIZE]
    print(f"Downloading stocks {start + 1}–{start + len(batch)}")
    download_batch(
        tickers=batch,
        output_dir=STOCK_RAW_DIR,
        asset_type="stock"
    )


# 5. DOWNLOAD MARKET AND MACRO DATA
download_batch(
    tickers=list(macro_tickers.keys()),
    output_dir=MACRO_RAW_DIR,
    asset_type="market_macro"
)

# saving the manifest
manifest = pd.DataFrame(download_log)
manifest_path = PROJECT_ROOT / "data" / "raw" / "download_manifest.csv"
manifest.to_csv(manifest_path, index=False)


print("\nDOWNLOAD SUMMARY")

print(manifest.groupby(
    ["asset_type", "status"]
).size())

print("\nFailed downloads:")

display(
    manifest[manifest["status"] == "failed"]
)

print("\nManifest saved:", manifest_path)


Stocks: 50
Additional market series: 5


/Users/srisuphachawla/Documents/DataScience_Projects/Indian_Equity_Market_Risk_Forecaster/.venv/lib/python3.13/site-packages/yfinance/utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
/Users/srisuphachawla/Documents/DataScience_Projects/Indian_Equity_Market_Risk_Forecaster/.venv/lib/python3.13/site-packages/yfinance/utils.py:585: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  dividends.set_index("date", inplace=True)
/Users/srisuphachawla/Documents/DataScience_Projects/Indian_Equity_Market_Risk_Forecaster/.venv/lib/python3.13/site-packages/yfinance/utils.py:587: Pandas4Warning: The inplace keyword

/Users/srisuphachawla/Documents/DataScience_Projects/Indian_Equity_Market_Risk_Forecaster/.venv/lib/python3.13/site-packages/yfinance/utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
/Users/srisuphachawla/Documents/DataScience_Projects/Indian_Equity_Market_Risk_Forecaster/.venv/lib/python3.13/site-packages/yfinance/utils.py:585: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  dividends.set_index("date", inplace=True)
/Users/srisuphachawla/Documents/DataScience_Projects/Indian_Equity_Market_Risk_Forecaster/.venv/lib/python3.13/site-packages/yfinance/utils.py:587: Pandas4Warning: The inplace keyword

/Users/srisuphachawla/Documents/DataScience_Projects/Indian_Equity_Market_Risk_Forecaster/.venv/lib/python3.13/site-packages/yfinance/utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
/Users/srisuphachawla/Documents/DataScience_Projects/Indian_Equity_Market_Risk_Forecaster/.venv/lib/python3.13/site-packages/yfinance/utils.py:585: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  dividends.set_index("date", inplace=True)
/Users/srisuphachawla/Documents/DataScience_Projects/Indian_Equity_Market_Risk_Forecaster/.venv/lib/python3.13/site-packages/yfinance/utils.py:587: Pandas4Warning: The inplace keyword

/Users/srisuphachawla/Documents/DataScience_Projects/Indian_Equity_Market_Risk_Forecaster/.venv/lib/python3.13/site-packages/yfinance/utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
/Users/srisuphachawla/Documents/DataScience_Projects/Indian_Equity_Market_Risk_Forecaster/.venv/lib/python3.13/site-packages/yfinance/utils.py:585: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  dividends.set_index("date", inplace=True)
/Users/srisuphachawla/Documents/DataScience_Projects/Indian_Equity_Market_Risk_Forecaster/.venv/lib/python3.13/site-packages/yfinance/utils.py:587: Pandas4Warning: The inplace keyword

/Users/srisuphachawla/Documents/DataScience_Projects/Indian_Equity_Market_Risk_Forecaster/.venv/lib/python3.13/site-packages/yfinance/utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
/Users/srisuphachawla/Documents/DataScience_Projects/Indian_Equity_Market_Risk_Forecaster/.venv/lib/python3.13/site-packages/yfinance/utils.py:585: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  dividends.set_index("date", inplace=True)
/Users/srisuphachawla/Documents/DataScience_Projects/Indian_Equity_Market_Risk_Forecaster/.venv/lib/python3.13/site-packages/yfinance/utils.py:587: Pandas4Warning: The inplace keyword


DOWNLOAD SUMMARY
asset_type    status 
market_macro  success     5
stock         failed      1
              success    49
dtype: int64

Failed downloads:


/Users/srisuphachawla/Documents/DataScience_Projects/Indian_Equity_Market_Risk_Forecaster/.venv/lib/python3.13/site-packages/yfinance/utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
/Users/srisuphachawla/Documents/DataScience_Projects/Indian_Equity_Market_Risk_Forecaster/.venv/lib/python3.13/site-packages/yfinance/multi.py:220: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  data.rename(columns=ctx.isins, inplace=True)


,ticker,asset_type,status,rows,first_date,last_date,file,error
3,ASIANPAINT.NS,stock,failed,0,NaT,NaT,NaN,No usable data returned.



Manifest saved: /Users/srisuphachawla/Documents/DataScience_Projects/Indian_Equity_Market_Risk_Forecaster/data/raw/download_manifest.csv


In [5]:
ticker = "ASIANPAINT.NS"

safe_name = ticker.replace("^", "").replace("=", "_").replace(".", "_")
output_path = STOCK_RAW_DIR / f"{safe_name}.parquet"

# Confirm the downloaded file exists
if not output_path.exists():
    raise FileNotFoundError(
        f"Downloaded file not found: {output_path}"
    )

# Read the saved data
retry_data = pd.read_parquet(output_path)

if retry_data.empty:
    raise ValueError("The saved Parquet file is empty.")

# Update manifest
manifest_path = PROJECT_ROOT / "data" / "raw" / "download_manifest.csv"
manifest = pd.read_csv(manifest_path)

mask = manifest["ticker"] == ticker

manifest.loc[mask, "status"] = "success"
manifest.loc[mask, "rows"] = len(retry_data)
manifest.loc[mask, "first_date"] = str(retry_data.index.min().date())
manifest.loc[mask, "last_date"] = str(retry_data.index.max().date())
manifest.loc[mask, "file"] = str(output_path)
manifest.loc[mask, "error"] = None

manifest.to_csv(manifest_path, index=False)

print("Manifest updated.")
print("Ticker:", ticker)
print("Rows:", len(retry_data))
print("First date:", retry_data.index.min().date())
print("Last date:", retry_data.index.max().date())

display(manifest.groupby(["asset_type", "status"]).size())

Manifest updated.
Ticker: ASIANPAINT.NS
Rows: 4159
First date: 2010-01-04
Last date: 2026-11-06


asset_type    status 
market_macro  success     5
stock         success    50
dtype: int64

In [8]:
manifest_path = PROJECT_ROOT / "data" / "raw" / "download_manifest.csv"
manifest = pd.read_csv(manifest_path)

audit_results = []

# Audit each dataset 
for _, item in manifest.iterrows():

    file_path = Path(item["file"]) if pd.notna(item["file"]) else None

    result = {
        "ticker": item["ticker"],
        "asset_type": item["asset_type"],
        "file_exists": False,
        "rows": 0,
        "first_date": None,
        "last_date": None,
        "duplicate_dates": None,
        "missing_values": None,
        "invalid_prices": None,
        "invalid_ohlc": None,
        "negative_volume": None,
        "audit_status": "failed"
    }

    try:
        if file_path is None or not file_path.exists():
            raise FileNotFoundError("Parquet file not found.")

        data = pd.read_parquet(file_path)

        if data.empty:
            raise ValueError("Dataset is empty.")

        result["file_exists"] = True
        result["rows"] = len(data)

        # Date coverage
        dates = pd.to_datetime(data.index, errors="coerce")

        result["first_date"] = str(dates.min().date())
        result["last_date"] = str(dates.max().date())

        # Duplicate dates
        result["duplicate_dates"] = int(data.index.duplicated().sum())

        # Missing values by column
        missing = data.isna().sum()
        result["missing_values"] = json.dumps(
            missing[missing > 0].to_dict()
        )

        # Price validation
        price_columns = ["Open", "High", "Low", "Close"]
        available_prices = [
            col for col in price_columns if col in data.columns
        ]

        if "Close" not in data.columns:
            raise ValueError("Close price column is missing.")

        prices = data[available_prices]

        result["invalid_prices"] = int(
            (prices <= 0).any(axis=1).sum()
        )

        # OHLC consistency: High must be >= all other prices,
        # and Low must be <= all other prices.
        if all(col in data.columns for col in price_columns):

            invalid_ohlc = (
                (data["High"] < data["Low"])
                | (data["High"] < data["Open"])
                | (data["High"] < data["Close"])
                | (data["Low"] > data["Open"])
                | (data["Low"] > data["Close"])
            )

            result["invalid_ohlc"] = int(invalid_ohlc.sum())

        # Volume validation
        if "Volume" in data.columns:
            result["negative_volume"] = int(
                (data["Volume"] < 0).sum()
            )

        result["audit_status"] = "success"

    except Exception as error:
        result["error"] = str(error)

    audit_results.append(result)


# Audit Report
audit = pd.DataFrame(audit_results)

audit_path = PROJECT_ROOT / "data" / "raw" / "data_quality_audit.csv"
audit.to_csv(audit_path, index=False)


# Results
print("AUDIT SUMMARY")
print(audit["audit_status"].value_counts())

print("\nDATASET COVERAGE")
display(
    audit[
        [
            "ticker",
            "asset_type",
            "rows",
            "first_date",
            "last_date"
        ]
    ]
)

print("\nDATA QUALITY ISSUES")

issue_columns = [
    "duplicate_dates",
    "invalid_prices",
    "invalid_ohlc",
    "negative_volume"
]

display(
    audit[
        audit[issue_columns].fillna(0).gt(0).any(axis=1)
        | audit["missing_values"].fillna("{}").ne("{}")
        | audit["audit_status"].ne("success")
    ]
)

print("\nFull audit saved:", audit_path)

AUDIT SUMMARY
audit_status
success    55
Name: count, dtype: int64

DATASET COVERAGE


,ticker,asset_type,rows,first_date,last_date
0,ADANIENT.NS,stock,4159,2010-01-04,2026-11-06
1,ADANIPORTS.NS,stock,4159,2010-01-04,2026-11-06
2,APOLLOHOSP.NS,stock,4159,2010-01-04,2026-11-06
3,ASIANPAINT.NS,stock,4159,2010-01-04,2026-11-06
4,AXISBANK.NS,stock,4159,2010-01-04,2026-11-06
5,BSE.NS,stock,4159,2010-01-04,2026-11-06
6,BAJAJ-AUTO.NS,stock,4159,2010-01-04,2026-11-06
7,BAJFINANCE.NS,stock,4159,2010-01-04,2026-11-06
8,BAJAJFINSV.NS,stock,4159,2010-01-04,2026-11-06
9,BEL.NS,stock,4159,2010-01-04,2026-11-06



DATA QUALITY ISSUES


,ticker,asset_type,file_exists,rows,first_date,last_date,duplicate_dates,missing_values,invalid_prices,invalid_ohlc,negative_volume,audit_status
5,BSE.NS,stock,True,4159,2010-01-04,2026-11-06,0,"{""Open"": 1748, ""High"": 1748, ""Low"": 1748, ""Clo...",0,0,0,success
10,BHARTIARTL.NS,stock,True,4160,2010-01-04,2026-11-06,0,"{""Open"": 1, ""High"": 1, ""Low"": 1, ""Close"": 1, ""...",0,0,0,success
11,CIPLA.NS,stock,True,4160,2010-01-04,2026-11-06,0,"{""Open"": 1, ""High"": 1, ""Low"": 1, ""Close"": 1, ""...",0,0,0,success
12,COALINDIA.NS,stock,True,4160,2010-01-04,2026-11-06,0,"{""Open"": 212, ""High"": 212, ""Low"": 212, ""Close""...",0,0,0,success
13,DRREDDY.NS,stock,True,4160,2010-01-04,2026-11-06,0,"{""Open"": 1, ""High"": 1, ""Low"": 1, ""Close"": 1, ""...",0,0,0,success
14,EICHERMOT.NS,stock,True,4160,2010-01-04,2026-11-06,0,"{""Open"": 1, ""High"": 1, ""Low"": 1, ""Close"": 1, ""...",0,0,0,success
15,ETERNAL.NS,stock,True,4160,2010-01-04,2026-11-06,0,"{""Open"": 2851, ""High"": 2851, ""Low"": 2851, ""Clo...",0,0,0,success
16,GRASIM.NS,stock,True,4160,2010-01-04,2026-11-06,0,"{""Open"": 1, ""High"": 1, ""Low"": 1, ""Close"": 1, ""...",0,0,0,success
18,HDFCBANK.NS,stock,True,4160,2010-01-04,2026-11-06,0,"{""Open"": 1, ""High"": 1, ""Low"": 1, ""Close"": 1, ""...",0,0,0,success
19,HDFCLIFE.NS,stock,True,4160,2010-01-04,2026-11-06,0,"{""Open"": 1944, ""High"": 1944, ""Low"": 1944, ""Clo...",0,0,0,success



Full audit saved: /Users/srisuphachawla/Documents/DataScience_Projects/Indian_Equity_Market_Risk_Forecaster/data/raw/data_quality_audit.csv


In [7]:
# Inspecting Invalid OHLC Observations 

problem_tickers = ["INR=X", "BZ=F"]

for ticker in problem_tickers:
    safe_name = ticker.replace("^", "").replace("=","_").replace(".", "_")

    if ticker == "INR-X":
        file_path = MACRO_RAW_DIR / f"{safe_name}.parquet"
    else: 
        file_path = MACRO_RAW_DIR / f"{safe_name}.parquet"

    data = pd.read_parquet(file_path)

    # we have already defined invalid_ohlc above

    print(f"\n{ticker}: {invalid_ohlc.sum()} invalid observations")

    display(
        data.loc[
            invalid_ohlc,
            ["Open", "High", "Low", "Close", "Volume"]
        ].head(15)
    )


INR=X: 0 invalid observations


Price,Open,High,Low,Close,Volume
Date,,,,,



BZ=F: 0 invalid observations


Price,Open,High,Low,Close,Volume
Date,,,,,
